# 02 - Exploratory Data Analysis (EDA)

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** charts aur correlations se samajhna ki kaun se features placement aur salary se judey hain.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries import aur data load karna
`matplotlib` charts ke liye. Salary ke charts me ek jagah **sirf Placed** students lete hain, kyunki Not Placed ka salary 0 hai aur wo distribution ko bigaad deta hai.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw = pd.read_csv('../datasets/studentplacement.csv')
placed = raw[raw['placement_status'] == 'Placed'].copy()
print('All students:', len(raw), '| Placed students:', len(placed))

## 2. Placed vs Not Placed
Pehle target class ka balance dekhte hain. Agar class bahut imbalanced hoti to alag treatment chahiye hota.

In [ ]:
counts = raw['placement_status'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(counts.index, counts.values)
ax.set_title('Placed vs Not Placed')
ax.set_ylabel('Students')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.show()

## 3. CGPA aur Salary ka distribution
CGPA poore dataset ka, aur Salary sirf Placed students ka histogram.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(raw['cgpa'], bins=30)
axes[0].set_title('CGPA Distribution (all students)')
axes[0].set_xlabel('CGPA')
axes[1].hist(placed['salary_package_lpa'], bins=30)
axes[1].set_title('Salary Distribution (Placed only)')
axes[1].set_xlabel('Salary (LPA)')
plt.tight_layout()
plt.show()

## 4. Saare numeric features ka distribution
Har numeric feature ka histogram, taaki shape (skew, uniform, normal) aur range dikhe.

In [ ]:
num_cols = [c for c in raw.select_dtypes(include='number').columns if c not in ['student_id', 'salary_package_lpa']]
fig, axes = plt.subplots(4, 5, figsize=(18, 11))
for ax, c in zip(axes.ravel(), num_cols):
    ax.hist(raw[c], bins=25)
    ax.set_title(c, fontsize=9)
for ax in axes.ravel()[len(num_cols):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Categorical features vs Placement rate
Har category me kitne % students Placed hue. Agar koi category ka rate alag hota to wo feature placement ke liye useful hota.

In [ ]:
cat_cols = ['gender', 'branch', 'college_tier', 'volunteer_experience']
rates = {c: raw.groupby(c)['placement_status'].apply(lambda s: (s == 'Placed').mean() * 100).round(2) for c in cat_cols}
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, c in zip(axes, cat_cols):
    ax.bar(rates[c].index, rates[c].values)
    ax.set_title('Placement rate by ' + c)
    ax.set_ylabel('% placed')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
for c in cat_cols:
    print(rates[c].to_dict())

## 6. Branch aur College Tier ke hisaab se Salary
Placed students ka average salary. (Dashboard ke home page par jo branch-wise average salary dikhta hai, wo Not Placed ke 0 salary ko bhi include karta hai, isliye wahan values lagbhag 7.2 aati hain; yahan sirf Placed liye hain.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
b = placed.groupby('branch')['salary_package_lpa'].mean().sort_values(ascending=False)
t = placed.groupby('college_tier')['salary_package_lpa'].mean()
axes[0].bar(b.index, b.values); axes[0].set_title('Avg salary by branch (Placed)'); axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(t.index, t.values); axes[1].set_title('Avg salary by college tier (Placed)')
for ax in axes:
    ax.set_ylabel('Avg salary (LPA)')
plt.tight_layout()
plt.show()
print(b.round(3).to_dict())
print(t.round(3).to_dict())

## 7. Numeric features ka Salary se correlation
Placed students me har numeric feature ka salary ke saath Pearson correlation. Yahi batata hai kaun se features salary predict karne me kaam aayenge.

In [ ]:
corr = placed.select_dtypes(include='number').corr()['salary_package_lpa'].drop('salary_package_lpa').sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(corr.index, corr.values)
ax.set_title('Correlation with salary (Placed students)')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.show()
corr.round(3).sort_values(ascending=False).head(6)

## 8. Top features vs Salary (scatter aur box plot)
Correlation me jo features upar aaye (`cgpa`, `coding_skill_score`) unka salary se relation, aur `internships_count` ke hisaab se salary ka spread.

In [ ]:
s = placed.sample(5000, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(s['cgpa'], s['salary_package_lpa'], alpha=0.2, s=8); axes[0].set_title('CGPA vs Salary'); axes[0].set_xlabel('CGPA')
axes[1].scatter(s['coding_skill_score'], s['salary_package_lpa'], alpha=0.2, s=8); axes[1].set_title('Coding Skill vs Salary'); axes[1].set_xlabel('Coding skill score')
groups = [g['salary_package_lpa'].values for _, g in placed.groupby('internships_count')]
axes[2].boxplot(groups); axes[2].set_xticklabels(sorted(placed['internships_count'].unique())); axes[2].set_title('Salary by internships count'); axes[2].set_xlabel('Internships')
for ax in axes:
    ax.set_ylabel('Salary (LPA)')
plt.tight_layout()
plt.show()

## 9. Correlation heatmap
Saare numeric columns ek saath: features aapas me kitne related hain (multicollinearity check).

In [ ]:
cm = raw.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.columns, fontsize=7)
plt.colorbar(im)
ax.set_title('Correlation heatmap')
plt.tight_layout()
plt.show()

## 10. Kaun se features Placement decide karte hain?
Placed aur Not Placed students ka numeric features ka average. Jis feature me bada fark ho wo placement se judaa hai.

In [ ]:
diff = raw.groupby('placement_status').mean(numeric_only=True).T
diff['difference_%'] = ((diff['Placed'] - diff['Not Placed']) / diff['Not Placed'] * 100).round(2)
diff.drop('salary_package_lpa').round(3).sort_values('difference_%', key=abs, ascending=False)

## 11. Outliers check (IQR method)
Har numeric feature me IQR ke bahar kitni values hain. Ye sirf jaanne ke liye hai; outliers hataye nahi gaye (tree models outliers se kam prabhavit hote hain).

In [ ]:
rows = []
for c in num_cols + ['salary_package_lpa']:
    q1, q3 = raw[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = int(((raw[c] < q1 - 1.5 * iqr) | (raw[c] > q3 + 1.5 * iqr)).sum())
    rows.append((c, n, round(n / len(raw) * 100, 2)))
pd.DataFrame(rows, columns=['feature', 'outliers', 'percent']).sort_values('outliers', ascending=False).head(8)

## 12. EDA ke findings
SEE_RESULTS